In [1]:
num_particles = 100
run_time_days = 20
time_step_minutes = 20
out_put_step_hours = 6

#initial position
lon0 = -50
lon1 = -48
lat0 = 1
lat1 = -0.5

depth_min = 1 #todo: figure near-surface depths 
depth_max = 10

offset = 1
ref_date = "1997-01-01"
#reproducibility
rdm_seed = 999

#paths
pathUVW= '/work/bk1450/b383184/Amazon/Mercator/data/variables'
# pathW= '/work/bk1450/b383184/Amazon/Atlantic/data/W'
pathTS = "/work/bk1450/b383184/Amazon/Mercator/data/variables"
Hgr = "/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc"
Zgr = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"

In [2]:
import numpy as np

In [3]:
out_path = f'../data/tracks_{rdm_seed}/' #path to store the particle zarr

start_time = np.datetime64(ref_date) + np.timedelta64(offset, "D")
start_time

np.datetime64('1997-01-02')

## Particles from the Plume to the Atlantic

* Release particles from the plume every 5 days for 4 years (1993-1999)
* Release time 1993 to 2013
* Number of particles =  10_000
* Release depth = (0,10)
* Tracking salinity and temperature

In [4]:
from parcels import ParticleSet
# from parcels import JITParticle
from parcels import ScipyParticle
from parcels import AdvectionRK4_3D
from parcels import AdvectionRK4
from parcels import Variable
from datetime import timedelta
import numpy as np
from parcels import FieldSet
from glob import glob

In [5]:
np.random.seed(rdm_seed)

### Copernicus Data (C grid)

In [6]:
# ufiles = sorted(glob(f"{pathUVW}/U_1997.nc"))
# vfiles = sorted(glob(f"{pathUVW}/V_1997.nc"))
# wfiles = sorted(glob(f"{pathUVW}/W_1997.nc"))
# Tfiles = sorted(glob(f"{pathTS}/T_1997.nc"))
# Sfiles = sorted(glob(f"{pathTS}/S_1997.nc"))

ufiles = f"{pathUVW}/U_1997-01.nc"
vfiles = f"{pathUVW}/V_1997-01.nc"
wfiles = f"{pathUVW}/W_1997-01.nc"
Tfiles = f"{pathTS}/T_1997-01.nc"
Sfiles = f"{pathTS}/S_1997-01.nc"

In [7]:
# print(len(ufiles))
# print(len(vfiles))
# print(len(wfiles))
# print(len(Tfiles))
# print(len(Sfiles))
ufiles

'/work/bk1450/b383184/Amazon/Mercator/data/variables/U_1997-01.nc'

In [8]:
## define the fieldset
filenames = {
    "U": {
        "data":ufiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
    
    "V": {
        "data": vfiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
    
    "W": {
        "data": wfiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
}

variables = {
    "U": "vozocrtx",
    "V": "vomecrty",
    "W": "vovecrtz",
}

interp_method = {
    "U":"cgrid_velocity",
    "V":"cgrid_velocity",
    "W":"cgrid_velocity",
}

# dimensions = {
#     "U": {"lon": "glamf", "lat": "gphif", "depth": "hdepw", "time": "time_counter"},
#     "V": {"lon": "glamf", "lat": "gphif", "depth": "hdepw", "time": "time_counter"},
#     "W": {"lon": "glamf", "lat": "gphif", "depth": "hdepw", "time": "time_counter"},
#     "T": {"lon": "glamf", "lat": "gphif", "depth": "hdept", "time": "time_counter"},
#     "S": {"lon": "glamf", "lat": "gphif", "depth": "hdept", "time": "time_counter"},
# }

dimensions = {
    "U": {"lon": "glamf", "lat": "gphif", "depth": "depthw", "time": "time_counter"},
    "V": {"lon": "glamf", "lat": "gphif", "depth": "depthw", "time": "time_counter"},
    "W": {"lon": "glamf", "lat": "gphif", "depth": "depthw", "time": "time_counter"},
}


## now the fieldset
fieldset = FieldSet.from_netcdf(
    filenames, 
    variables, 
    dimensions,
    interp_method = interp_method,
    # tracer_interp_method="cgrid_tracer",
    deferred_load=True,
    allow_time_extrapolation = False,
    gridindexingtype = "nemo",
)

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vomecrty' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vovecrtz' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [9]:
fieldset.U

<Field>
    name            : 'U'
    grid            : CurvilinearSGrid(lon=array([[-94.96, -94.88, -94.79, ...,  9.79,  9.88,  9.96],
       [-94.96, -94.88, -94.79, ...,  9.79,  9.88,  9.96],
       [-94.96, -94.88, -94.79, ...,  9.79,  9.88,  9.96],
       ...,
       [-94.96, -94.88, -94.80, ...,  9.80,  9.89,  9.97],
       [-94.96, -94.88, -94.80, ...,  9.81,  9.89,  9.97],
       [-94.96, -94.88, -94.80, ...,  9.81,  9.89,  9.97]], shape=(499, 1260), dtype=float32), lat=array([[-9.91, -9.91, -9.91, ..., -9.91, -9.91, -9.91],
       [-9.83, -9.83, -9.83, ..., -9.83, -9.83, -9.83],
       [-9.74, -9.74, -9.74, ..., -9.74, -9.74, -9.74],
       ...,
       [ 29.80,  29.80,  29.80, ...,  29.84,  29.84,  29.84],
       [ 29.87,  29.87,  29.87, ...,  29.91,  29.91,  29.91],
       [ 29.94,  29.94,  29.94, ...,  29.98,  29.98,  29.98]], shape=(499, 1260), dtype=float32), time=array([ 0.00,  86400.00,  172800.00, ...,  2419200.00,  2505600.00,  2592000.00], shape=(31,)), time_origin=19

In [10]:
start_pos_along_line = np.random.uniform(0,1,size=num_particles)
start_lon = lon0 + start_pos_along_line * (lon1-lon0)
start_lat = lat0 + start_pos_along_line * (lat1-lat0)
start_depth = np.random.uniform(depth_min,depth_max,size=num_particles)
start_times = np.datetime64(start_time)

In [11]:
pset = ParticleSet(
    fieldset=fieldset,
    lon = start_lon,
    lat = start_lat,
    depth=start_depth,
    time=start_times
) 

out_fn = f'Parcels_run_{rdm_seed}_{start_time}.zarr'

output_file = pset.ParticleFile(
    name=out_path+out_fn,
    outputdt=timedelta(hours=out_put_step_hours),
    chunks = (num_particles,int(run_time_days*24/out_put_step_hours/4))
)

In [12]:
##check the error
def CheckError(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

In [ ]:
## Execute particles
pset.execute(
    [AdvectionRK4_3D,CheckError],
    runtime=timedelta(days=run_time_days),
    dt=timedelta(minutes=time_step_minutes),
    output_file= output_file
)

INFO: Output files are stored in ../data/tracks_999/Parcels_run_999_1997-01-02.zarr.
  0%|          | 0/1728000.0 [00:00<?, ?it/s]

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'vomecrty' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWar